# internal_wave1: internal wave beams from an oscillating point source
pyOM2: for_config/internal_wave1.f90, py_config/internal_wave1.py

Non-hydrostatic x-z slice (nh_simple), uniform stratification N0, a localised
zonal force oscillating with frequency om0 < N0 radiates wave beams at the angle
cos(theta) = om0/N0 to the vertical.

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_simple.model import model

In [ ]:
fac        = 2.0
nx, ny, nz = int(64*fac), 1, int(64*fac)
dx = dz    = 0.25/fac
dt         = 0.5/fac
N0         = 2*np.pi/10.            # stability frequency
om0        = 2*np.pi/15.            # forcing frequency
mix        = 5e-3/fac**2
nsteps     = 1200                   # 300 s
plot_every = 100

par = dict(nx=nx, ny=ny, nz=nz, dx=dx, dy=dx, dz=dz, dt=dt,
           f0=0., N0=N0, Ro=1., dsqr=1., periodic_in_z=False,
           Ah=mix, Av=mix, buoyancy_superbee_advection=True)

In [ ]:
class internal_wave1(model):

    def set_forcing(self, tau):
        self.time += self.dt
        self.du = OM.add_to_array(self.du, (tau,), self.u0*np.sin(om0*self.time))

In [ ]:
M = internal_wave1(par)
M.setup()
# local coordinates with halo points, arrays are (x,y,z), z = 0 at the surface
h = M.halo
x = (np.arange(M.Nx) - h + M.xs_pe - 0.5)*dx
y = (np.arange(M.Ny) - h + M.ys_pe - 0.5)*dx
z = (np.arange(M.Nz) - h + M.zs_pe - 0.5)*dz - nz*dz
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
x0, z0 = nx//2*dx, -nz//2*dz
M.u0 = OM.np.asarray(1./(100*60*dt)*np.exp(-(X + dx/2 - x0)**2/dx**2)*np.exp(-(Z - z0)**2/dz**2))
M.time = 0.

In [ ]:
def plot(t):
    clear_output(wait=True)
    w = M.w[h:-h, h, h:-h].T
    plt.figure(figsize=(7, 6))
    plt.pcolormesh(x[h:-h], z[h:-h], w, cmap='RdBu_r', vmin=-1e-4, vmax=1e-4)
    plt.colorbar(label='w [m/s]'); plt.xlabel('x [m]'); plt.ylabel('z [m]')
    plt.title('t = %.0f s' % t); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)